# Generate irAE-arm figures

End-to-end driver that builds the IPIO irAE-arm figures from the survival
pipeline outputs:

1. **Paired volcano** (`paired_volcano_irae.{png,pdf}`) -- univariate Cox at
   landmarks 0d and +90d, colored by clinical category.
2. **Discrimination vs. baseline** (`discrimination_vs_baseline_irae.{png,pdf}`)
   -- test mean AUC(t) and C-index for the full labs+genomics Cox model vs. a
   clinical-covariates-only baseline, at both landmarks.
3. **Model importance** (`model_importance_irae.{png,pdf}`) -- top Cox
   log-HR coefficients at both landmarks.

There is no PGS-forest figure for this project (see the final markdown cell).
Each figure section is independent once the shared `Imports`, `Paths`, and
`Clinical category mapping` cells have been run.

## Imports

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Reuse the shared RCPARAMS + parse_feature helpers from COMPASS rather than
# duplicating them here.
sys.path.insert(0, "/data/gusev/USERS/jpconnor/code/CAIA/COMPASS/survival_analysis")
from helpers.plotting import RCPARAMS, parse_feature

plt.rcParams.update(RCPARAMS)

## Paths

`BASE` points at the directory containing `cox/landmark_{lm}/...` outputs.
Override for local copies.

In [ ]:
BASE = Path(
    "/data/gusev/USERS/jpconnor/data/CAIA/IPIO/"
    "survival_analysis/local_runs"
)
OUT_DIR = Path("/data/gusev/USERS/jpconnor/figures/CAIA/IPIO")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANDMARKS = [0, 90]
TOP_N = 15  # number of features shown per importance panel

## Clinical category mapping

IPIO is a pan-cancer immunotherapy cohort, not prostate-specific, so this
taxonomy does **not** reuse COMPASS's `ANDROGEN = {"PSA", "Testosterone"}`
bucket. Instead it groups labs into CBC / CMP / LFT / Thyroid / Vitals /
Other, with a dedicated `Thyroid` bucket since thyroid dysfunction is one of
the most common irAEs. **This list is a placeholder** adapted from general
irAE-monitoring conventions, not from the actual canonical-lab list for this
cohort -- revisit once that's known.

In [ ]:
# NOTE: adapted for an immunotherapy / irAE cohort instead of COMPASS's
# prostate-specific taxonomy (no "Androgen axis" bucket here). Revisit this
# list once the real canonical-lab set for the IPIO cohort is confirmed.
CBC = {
    "WBC", "RBC", "Hemoglobin", "Hematocrit",
    "MCV", "MCH", "MCHC", "RDW", "Platelets",
    "Neutrophils absolute", "Lymphocytes absolute",
    "Monocytes absolute", "Eosinophils absolute", "Basophils absolute",
}
CMP = {"Sodium", "Potassium", "Chloride", "CO2",
       "BUN", "Creatinine", "Glucose", "Calcium"}
LFT = {"ALT", "AST", "Alkaline phosphatase",
       "Total bilirubin", "Direct bilirubin",
       "Albumin", "Globulin", "Total protein", "PT"}
THYROID = {"TSH", "Free T4"}  # thyroiditis is a common irAE -- own bucket
VITALS = {"Body weight", "Body temperature", "Heart rate",
          "Respiratory rate", "Systolic blood pressure",
          "Diastolic blood pressure"}
OTHER = {"CRP", "ESR"}  # + anything unmapped falls here by default

CATEGORY_MAP = {}
for label, members in [
    ("CBC", CBC), ("CMP", CMP), ("LFT", LFT),
    ("Thyroid", THYROID), ("Vitals", VITALS), ("Other", OTHER),
]:
    for m in members:
        CATEGORY_MAP[m] = label

DRAW_ORDER = ["Other", "Vitals", "Thyroid", "CMP", "LFT", "CBC"]
LEGEND_ORDER = ["CBC", "LFT", "CMP", "Thyroid", "Vitals", "Other"]

CATEGORY_COLORS = {
    "CBC":     "#16a085",
    "LFT":     "#e67e22",
    "CMP":     "#7d3c98",
    "Thyroid": "#8e1c2b",
    "Vitals":  "#5d6d7e",
    "Other":   "#95a5a6",
}
NS_COLOR = "#d5d8dc"


def assign_category(lab_name: str) -> str:
    return CATEGORY_MAP.get(lab_name, "Other")


def format_label(lab_name, feature_stat):
    if not feature_stat or pd.isna(feature_stat) or feature_stat == "":
        return lab_name
    return f"{lab_name} ({feature_stat})"

## Figure 1 -- Paired volcano (univariate Cox)

Each dot is one (lab × feature_stat) pair. ns features (q ≥ 0.05) are faded
gray; q-significant features are colored by their clinical category.

Source: `cox/landmark_{lm}/both/cox_agg_univariate_nobs_adjusted.csv`,
filtered to `endpoint == "irae"`.

In [ ]:
# ----------------------------- labeling knobs ---------------------------
TOP_K_PER_PANEL = 6
# Placeholder -- re-pick once real univariate hits are known for this cohort.
# LFT + thyroid labs are plausible irAE signals so used here as a stand-in.
ALWAYS_LABEL = {"TSH", "ALT", "Creatinine"}
LABEL_FORMAT = "{lab} ({stat})"
LABEL_FONTSIZE = 8.5
MIN_LABEL_SPACING_NLP = 0.55
PANEL_XLIM = (-1.5, 1.5)
Y_MAX_CAP = 30  # -log10(p) ceiling; anything above this is drawn at the cap as a triangle


def q_threshold_neglog10p(sub):
    """y-value (-log10 p) at which q just crosses 0.05; None if no q-sig features."""
    sig = sub.loc[sub["q_value"] < 0.05, "p_value"]
    if sig.empty:
        return None
    cutoff_p = float(sig.max())
    return float(-np.log10(max(cutoff_p, 1e-300)))


def _auto_label(ax, sub, *, top_k, always_label, label_format):
    """Stack labels at the left/right panel edges with leader lines.

    Selection rule: dedupe to the most-significant stat per lab, then include
    the `always_label` whitelist + top_k by p-value.
    """
    sig = sub.loc[sub["sig"]].copy()
    if sig.empty:
        return

    best = (sig.sort_values("p_value")
               .drop_duplicates("lab_name", keep="first"))
    always_sig = best.loc[best["lab_name"].isin(always_label)]
    extra = best.loc[~best["lab_name"].isin(always_label)].head(top_k)
    to_label = pd.concat([always_sig, extra]).drop_duplicates(
        subset=["lab_name", "feature_stat"])
    if to_label.empty:
        return

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]

    def _stack(side_df, side):
        if side_df.empty:
            return
        side_df = side_df.sort_values("neglog10p", ascending=False)
        if side == "left":
            label_x = xlim[0] + 0.04 * xspan
            ha = "left"
        else:
            label_x = xlim[1] - 0.04 * xspan
            ha = "right"
        last_y = None
        for _, r in side_df.iterrows():
            target_y = min(r["neglog10p"], Y_MAX_CAP)
            if last_y is not None and target_y > last_y - MIN_LABEL_SPACING_NLP:
                target_y = last_y - MIN_LABEL_SPACING_NLP
            if target_y < ylim[0] + 0.05 * yspan:
                continue
            last_y = target_y
            text = label_format.format(lab=r["lab_name"],
                                       stat=r["feature_stat"])
            color = CATEGORY_COLORS[r["category"]]
            ax.annotate(
                text, xy=(r["coef_feature"], min(r["neglog10p"], Y_MAX_CAP)),
                xytext=(label_x, target_y), textcoords="data",
                ha=ha, va="center",
                fontsize=LABEL_FONTSIZE, color=color, weight="semibold",
                arrowprops=dict(arrowstyle="-", lw=0.45, color="#95a5a6"),
                zorder=10,
            )

    left = to_label.loc[to_label["coef_feature"] < 0]
    right = to_label.loc[to_label["coef_feature"] >= 0]
    _stack(left, "left")
    _stack(right, "right")


def plot_volcano_panel(ax, sub, title):
    sub = sub.copy()
    sub["category"] = sub["lab_name"].map(assign_category)
    sub["neglog10p"] = -np.log10(sub["p_value"].clip(lower=1e-300))
    sub["sig"] = sub["q_value"] < 0.05
    # Cap extreme -log10(p) so a handful of huge values don't squash the rest.
    sub["capped"] = sub["neglog10p"] > Y_MAX_CAP
    sub["y"] = sub["neglog10p"].clip(upper=Y_MAX_CAP)

    ns = sub.loc[~sub["sig"]]
    ax.scatter(ns["coef_feature"], ns["y"],
               s=20, color=NS_COLOR, alpha=0.45,
               edgecolor="none", zorder=1)

    for cat in DRAW_ORDER:
        cat_sig = sub.loc[sub["sig"] & (sub["category"] == cat)]
        if cat_sig.empty:
            continue
        base_kw = dict(
            color=CATEGORY_COLORS[cat],
            alpha=0.92,
            edgecolor="white", linewidth=0.6,
        )
        in_range = cat_sig.loc[~cat_sig["capped"]]
        if not in_range.empty:
            ax.scatter(
                in_range["coef_feature"], in_range["y"],
                s=40, marker="o", zorder=3, **base_kw,
            )
        over = cat_sig.loc[cat_sig["capped"]]
        if not over.empty:
            ax.scatter(
                over["coef_feature"], over["y"],
                s=60, marker="^", zorder=4, **base_kw,
            )
            for _, r in over.iterrows():
                ax.annotate(
                    f"p={r['p_value']:.1e}",
                    xy=(r["coef_feature"], Y_MAX_CAP),
                    xytext=(0, -8), textcoords="offset points",
                    ha="center", va="top",
                    fontsize=7.5, color=CATEGORY_COLORS[cat],
                    zorder=7,
                )

    ax.set_xlim(*PANEL_XLIM)
    y_max = float(sub["y"].max()) if not sub.empty else 5.0
    ax.set_ylim(-0.2, max(y_max * 1.10, 5.0))

    ax.axvline(0, color="grey", linestyle="-", linewidth=0.7, zorder=0)
    for x in (-0.5, 0.5):
        ax.axvline(x, color="grey", linestyle="--", linewidth=0.6,
                   alpha=0.7, zorder=0)
    q_y = q_threshold_neglog10p(sub)
    if q_y is not None:
        ax.axhline(q_y, color="black", linestyle=":", linewidth=0.9, zorder=0)

    _auto_label(ax, sub,
                top_k=TOP_K_PER_PANEL,
                always_label=ALWAYS_LABEL,
                label_format=LABEL_FORMAT)

    ax.set_xlabel("Cox log HR per SD")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_title(title, fontsize=12.5, weight="bold")

    n_tested = len(sub)
    n_sig = int(sub["sig"].sum())
    breakdown = (sub.loc[sub["sig"], "category"]
                 .value_counts()
                 .reindex([c for c in LEGEND_ORDER if c != "Other"], fill_value=0))
    bd_str = "  ".join(f"{c} {int(n)}" for c, n in breakdown.items())
    ax.text(0.99, 0.02,
            f"{n_sig} / {n_tested} q<0.05   ·   {bd_str}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8.5, color="#5d6d7e", family="monospace")

In [ ]:
# Load both landmarks and tag with landmark_days (authoritative from path).
def _load_uni(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_univariate_nobs_adjusted.csv"
    df = pd.read_csv(p)
    df["landmark_days"] = landmark
    return df

uni = pd.concat([_load_uni(lm) for lm in LANDMARKS], ignore_index=True)
uni = uni.loc[uni["endpoint"] == "irae"].copy()
uni = uni.dropna(subset=["coef_feature", "p_value", "q_value"])
print(f"{len(uni)} (lab × stat) rows across landmarks "
      f"{sorted(uni['landmark_days'].unique())}")

# Filter unstable Cox estimates: |log HR| > 2 or CI spans > 2 orders of magnitude.
COEF_CAP = 2.0
CI_RATIO_CAP = 100
ci_ratio = uni["ci_upper"] / uni["ci_lower"]
mask = (uni["coef_feature"].abs() <= COEF_CAP) & (ci_ratio < CI_RATIO_CAP)
print(f"dropping {int((~mask).sum())} / {len(uni)} unstable rows")
uni = uni.loc[mask].copy()
print(f"{len(uni)} rows remaining")

panels = [(0, "0 days"), (90, "+90 days")]
fig, axes = plt.subplots(
    1, 2, figsize=(14, 6),
    sharex=True, sharey=False,
    gridspec_kw={"wspace": 0.08},
)
for ax, (lm, title) in zip(axes, panels):
    sub = uni.loc[uni["landmark_days"] == lm]
    if sub.empty:
        ax.text(0.5, 0.5, f"(no data for landmark = {lm}d)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_axis_off()
        continue
    plot_volcano_panel(ax, sub, title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
handles.append(Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=NS_COLOR, markersize=8,
                      label="ns (q ≥ 0.05)"))
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"paired_volcano_irae.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

## Figure 2 -- Discrimination vs. baseline (AUC + C-index)

Two panels -- **mean IPCW AUC(t)** and **C-index** on the held-out test fold
-- comparing the full labs+genomics Cox model against its
**clinical-covariates-only baseline** (GENDER/AGE/pd1pdl1/ctla4, hatched), on
the full cohort, at both landmarks. XGBoost is out of scope for IPIO, so
(unlike COMPASS) this is Cox-only.

Sources (irae endpoint):
- **Cox labs** `cox/landmark_{lm}/both/cox_agg_multivariable_metrics.csv` → `test_mean_auc_t`, `test_c_index`
- **Cox baseline** `cox/landmark_{lm}/baseline/cox_agg_baseline_metrics.csv` → `test_mean_auc_t`, `test_c_index`

In [ ]:
# Full labs+genomics Cox model vs. its clinical-covariates-only baseline, on the full cohort.
def _read_metrics(path, auc_col, cindex_col):
    """Return (mean_auc_t, c_index) for the irae endpoint, or (nan, nan)."""
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        return (np.nan, np.nan)
    row = df.loc[df["endpoint"] == "irae"]
    if row.empty:
        return (np.nan, np.nan)
    row = row.iloc[0]
    return (float(row[auc_col]), float(row[cindex_col]))


def cox_labs(lm):
    return _read_metrics(BASE / "cox" / f"landmark_{lm}" / "both" / "cox_agg_multivariable_metrics.csv",
                         "test_mean_auc_t", "test_c_index")


def cox_baseline(lm):
    return _read_metrics(BASE / "cox" / f"landmark_{lm}" / "baseline" / "cox_agg_baseline_metrics.csv",
                         "test_mean_auc_t", "test_c_index")


# (label, loader, color, hatch). Baseline is the hatched, lighter twin.
SERIES = [
    ("Elastic-Net Cox (labs + genomics)", cox_labs,     "#4C72B0", None),
    ("Cox baseline (clinical covariates)", cox_baseline, "#9DB3D6", "//"),
]

# data[label][k] = (auc, cindex) at LANDMARKS[k]
data = {name: [loader(lm) for lm in LANDMARKS] for name, loader, _, _ in SERIES}

n_series = len(SERIES)
bar_width = 0.32
x_positions = np.arange(len(LANDMARKS))
offsets = (np.arange(n_series) - (n_series - 1) / 2) * bar_width

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, (ylabel, idx) in zip(axes, [("Test Mean AUC(t)", 0), ("Test C-index", 1)]):
    for i, (name, _loader, color, hatch) in enumerate(SERIES):
        vals = [data[name][k][idx] for k in range(len(LANDMARKS))]
        bar_x = x_positions + offsets[i]
        ax.bar(bar_x, [v if np.isfinite(v) else 0.0 for v in vals], bar_width,
               color=color, hatch=hatch, edgecolor="white", linewidth=0.8, label=name)
        for x, v in zip(bar_x, vals):
            if np.isfinite(v):
                ax.text(x, v + 0.006, f"{v:.3f}", ha="center", va="bottom",
                        fontsize=7.5, weight="semibold", color=color)
    finite = [data[n][k][idx] for n in data for k in range(len(LANDMARKS))
              if np.isfinite(data[n][k][idx])]
    ax.set_ylim(0.45, min(1.0, (max(finite) if finite else 0.7) * 1.12))
    ax.set_xticks(x_positions)
    ax.set_xticklabels([f"{('+' if lm > 0 else '')}{lm} days" for lm in LANDMARKS])
    ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.9)
    ax.set_ylabel(ylabel)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3)

axes[0].legend(loc="upper right", fontsize=8, ncol=1)
fig.suptitle("Labs vs. baseline covariates — irAE · full cohort",
             fontsize=13, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])

for ext in ("png", "pdf"):
    out = OUT_DIR / f"discrimination_vs_baseline_irae.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

## Figure 3 -- Model importance (Cox)

COMPASS's 2×2 grid paired Cox coefficients with XGBoost gain; XGBoost is out
of scope for IPIO, so this simplifies to a single row -- one panel per
landmark of the top Cox log-HR coefficients. Baseline clinical covariates
(GENDER/AGE/pd1pdl1/ctla4) are excluded so the panel focuses on lab- and
genomics-derived features.

Source: `cox/landmark_{lm}/both/cox_agg_multivariable.csv` → `endpoint ==
"irae"`, column `coef`.

In [ ]:
def load_cox_coefs(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_multivariable.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "irae"].copy()
    # Exclude baseline clinical covariates (GENDER/AGE/pd1pdl1/ctla4) so the
    # panel focuses on lab- and genomics-derived features. The exact flag
    # column name depends on cox_multivariable.py's output schema -- check a
    # couple of likely names defensively.
    baseline_flag_col = next(
        (c for c in ("is_baseline_covariate", "is_age_covariate", "is_clinical_covariate")
         if c in df.columns),
        None,
    )
    if baseline_flag_col is not None:
        df = df.loc[~df[baseline_flag_col].fillna(False).astype(bool)]
    df = df.loc[df["coef"].fillna(0) != 0]
    return df


def render_importance_panel(ax, df, title):
    if df.empty:
        ax.text(0.5, 0.5, "(no features to display)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    df = df.copy()
    df["category"] = df["lab_name"].map(assign_category) if "lab_name" in df.columns else "Other"

    df = df.reindex(df["coef"].abs().sort_values(ascending=False).index).head(TOP_N)
    df = df.iloc[::-1]
    values = df["coef"].to_numpy()

    colors = [CATEGORY_COLORS.get(c, CATEGORY_COLORS["Other"]) for c in df["category"]]
    y = np.arange(len(df))
    ax.barh(y, values, color=colors, edgecolor="white", linewidth=0.6)
    ax.axvline(0, color="black", linewidth=0.5, zorder=0)

    if "feature_stat" in df.columns:
        labels = [format_label(r["lab_name"], r["feature_stat"]) for _, r in df.iterrows()]
    elif "lab_name" in df.columns:
        labels = df["lab_name"].tolist()
    else:
        labels = df["feature"].tolist() if "feature" in df.columns else [str(i) for i in df.index]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlabel("log HR coefficient", fontsize=10)
    ax.set_title(title, fontsize=11, weight="bold")


fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
for ax, lm in zip(axes, LANDMARKS):
    sign = "+" if lm > 0 else ""
    title = f"Elastic-Net Cox  ·  {sign}{lm} days"
    try:
        df = load_cox_coefs(lm)
    except FileNotFoundError as e:
        ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=8, color="#c0392b")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        continue
    render_importance_panel(ax, df, title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"model_importance_irae.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

## Figure 4 -- PGS interpretation (out of scope)

COMPASS's Figure 4 plots PGS-only Cox HR forest plots for Testosterone and
PSA polygenic scores. IPIO has no PGS data for this cohort, so that figure is
omitted entirely here.